In [0]:
# Gold Layer: Aggregated, business-ready tables
from pyspark.sql import functions as F

# ------------------------------------------------------------------
# Config — update these to match your catalog/schema and table names
# ------------------------------------------------------------------
catalog = "workspace"
silver_schema = "silver"
gold_schema = "gold"
source_table = f"{catalog}.{silver_schema}.employee"
target_table = f"{catalog}.{gold_schema}.employee_dept_agg"

# ------------------------------------------------------------------
# Read silver table
# ------------------------------------------------------------------
silver_df = spark.table(source_table)

# ------------------------------------------------------------------
# Gold transformation: department-level aggregation
# (adjust dimensions / measures to your data)
# ------------------------------------------------------------------
gold_df = (
    silver_df
    .groupBy("dept")
    .agg(
        F.countDistinct("empno").alias("employee_count"),
        F.sum("salary").alias("total_salary"),
        F.avg("salary").alias("avg_salary"),
        F.min("salary").alias("min_salary"),
        F.max("salary").alias("max_salary"),
        F.max("modifiedat").alias("last_modified_at"),
    )
)

# ------------------------------------------------------------------
# Write to gold as a managed Delta table (idempotent MERGE upsert)
# ------------------------------------------------------------------
gold_df.createOrReplaceTempView("gold_updates")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {target_table} (
  dept STRING,
  employee_count BIGINT,
  total_salary DECIMAL(18,2),
  avg_salary DECIMAL(18,2),
  min_salary DECIMAL(18,2),
  max_salary DECIMAL(18,2),
  last_modified_at TIMESTAMP
)
USING DELTA
""")

spark.sql(f"""
MERGE INTO {target_table} AS t
USING gold_updates AS s
  ON  t.dept = s.dept
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

display(spark.table(target_table))
